# Test Environment

Checks the gymnasium environment and verifies IMU orientation. Press `shift+enter` to run each cell.

In [1]:
# Import standard libraries
import math
from pathlib import Path
import sys
import time

# Third-party libraries
from gymnasium.utils.env_checker import check_env
import matplotlib.pyplot as plt
import mujoco
import numpy as np

/config/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-wqjkeurw because there was an issue with the default path (/config/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# Import the custom environment
from balance_bot_env_dr import BalanceBotEnv, DomainRandomConfig

In [3]:
# Settings
MJCF_PATH = Path("../../mechanical/bala-c-plus-simplified/bala-c-plus-simplified.xml")
SEED = 42
CHASSIS_PITCH_TRIM_DEG = 11.4

## Check that the environment loads

In [4]:
# Initialize environment
env = BalanceBotEnv(
    mjcf_path=MJCF_PATH, 
    render_mode="human", 
    chassis_trim_deg=CHASSIS_PITCH_TRIM_DEG)

# Print observation and action spaces
print(env.observation_space)
print(env.action_space)

# Print trim readings
print(f"_eq_chassis_rad  = {math.degrees(env._eq_chassis_rad):+.2f} deg   (expect +11.40)")
print(f"_pitch_trim_rad  = {math.degrees(env._pitch_trim_rad):+.2f} deg   (expect -11.40)")

Box([ -3.1415927 -20.         -5.        -20.        -10.        -12.566371 ], [ 3.1415927 20.         5.        20.        10.        12.566371 ], (6,), float32)
Box(-1.0, 1.0, (2,), float32)
_eq_chassis_rad  = +11.40 deg   (expect +11.40)
_pitch_trim_rad  = -11.40 deg   (expect -11.40)


In [5]:
# Use gymnasium's built-in environment checker
# Ignore the warning, error is bad
check_env(env)
env.close()

eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4
eq=11.4 dr_range=none start_chassis=11.4


/opt/rl-env/lib/python3.12/site-packages/gymnasium/utils/env_checker.py:434: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


## Run simulation with dummy policy

In [ ]:
# Initialize environment
env = BalanceBotEnv(
    mjcf_path=MJCF_PATH, 
    render_mode="human", 
    chassis_trim_deg=CHASSIS_PITCH_TRIM_DEG
)

In [ ]:
# Reset environment and print observation
obs, info = env.reset(seed=SEED)
print(obs)

In [ ]:
# Confirm pitch_error is ~0 at reset (reward centered correctly)
w,x,y,z = env.data.sensor("imu_orientation").data
pt = math.atan2(1-2*(x*x+y*y), -2*(y*z+w*x))
print(f"pitch_error at reset:  {math.degrees(pt - env._pitch_trim_rad):+.2f} deg   (expect ~0)")

In [ ]:
# Manual step with no action (don't turn the motors)
obs, reward, terminated, truncated, info = env.step(np.array([0.0, 0.0]))
print(f"obs: {obs}")
print(f"reward: {reward}")
print(f"terminated: {terminated}")
print(f"truncated: {truncated}")

In [ ]:
# Test: random policy
obs, _ = env.reset()
total_reward = 0

# Go for 200 steps
terminated = False
terminated = False
for step in range(200):
    step_start = time.time()

    # Choose a random action
    action = env.action_space.sample()

    # Take a step with the given action
    obs, reward, terminated, truncated, _ = env.step(action)

    # Accumulate reward
    total_reward += reward

    # Render in MuJoCo
    env.render()

    # Show that we stopped
    if terminated or truncated:
        print(f"Episode ended at step {step}")
        break

    # Make sure the loop iteration time matches the MJCF timestep time (5 ms)
    # i.e. slow the simulation down so we can see the robot behave in real time
    slack = env.model.opt.timestep - (time.time() - step_start)
    if slack > 0:
        time.sleep(slack)

print(f"Final obs: {obs}")
print(f"Final reward: {total_reward}")

## Check IMU model

In [ ]:
def check(name, ok, detail=""):
    """Check results"""
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))
    return ok

In [ ]:
def pitch_true_of(d, sensor="imu_orientation"):
    """Set a known tilt, read pitch_true back."""
    # Exactly the computation in step()
    w, x, y, z = d.sensor(sensor).data
    up_y = 2.0*(y*z + w*x)
    up_z = 1.0 - 2.0*(x*x + y*y)
    return math.atan2(up_z, -up_y)

In [ ]:
print(f"{'commanded':>10} {'pitch_true':>11} {'expected':>9}")
worst = 0.0
for deg in [0, 5, -5, 10, -10, 20, -20, 29, -29]:
    env.reset(seed=0)
    th = math.radians(deg)
    env.data.qpos[2] = 1.0                                        # lift off the floor
    env.data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]   # rotate about +Y
    env.data.qvel[:] = 0.0
    mujoco.mj_forward(env.model, env.data)

    pt = math.degrees(pitch_true_of(env.data))
    worst = max(worst, abs(pt - (-deg)))
    print(f"{deg:>10.1f} {pt:>11.3f} {-deg:>9.1f}")

check("pitch_true matches commanded tilt (negated)", worst < 0.01,
      f"max err {worst:.4f} deg")

env.reset(seed=0)

In [ ]:
# Gyro: pitch is rotation about chassis Y, which maps to site X -> gyro[0].
# (The Bala2 code read gyro[1], which in this frame is yaw.)
env.reset(seed=0)
env.data.qvel[:] = 0.0
env.data.qvel[4] = 1.0                     # +1 rad/s about chassis Y
mujoco.mj_forward(env.model, env.data)
g_pitch = np.array(env.data.sensor(env.sensor_imu_gyro).data)

env.reset(seed=0)
env.data.qvel[:] = 0.0
env.data.qvel[5] = 1.0                     # +1 rad/s yaw
mujoco.mj_forward(env.model, env.data)
g_yaw = np.array(env.data.sensor(env.sensor_imu_gyro).data)

print(f"pitch 1 rad/s -> gyro = {np.round(g_pitch, 4)}")
print(f"yaw   1 rad/s -> gyro = {np.round(g_yaw, 4)}")

check("pitch rate lands on gyro[0]", abs(abs(g_pitch[0]) - 1.0) < 1e-6)
check("gyro[0] is negative for +theta (matches pitch_true sign)", g_pitch[0] < 0)
check("yaw does not leak into gyro[0]", abs(g_yaw[0]) < 1e-6)

In [ ]:
# Accelerometer: check it upright and at rest ON THE FLOOR, where gravity is
# balanced by the contact force. (In mid-air the accelerometer reads ~0, not g --
# it measures proper acceleration, so a free-falling robot reads nothing.)
env.reset(seed=0)
env.data.qvel[:] = 0.0
for _ in range(20):                        # let contacts settle
    env.data.ctrl[:] = 0.0
    mujoco.mj_step(env.model, env.data)
a = np.array(env.data.sensor(env.sensor_imu_accel).data)

print(f"\nupright on floor, accel = {np.round(a, 3)}")
check("gravity sits on IMU -Y", abs(a[1] + 9.81) < 0.5, f"a_y={a[1]:.3f}")
check("X channel ~0 (the axis the old code read)", abs(a[0]) < 0.5, f"a_x={a[0]:.3f}")

env.reset(seed=0)

## Check actuator model

In [ ]:
# Actuator + wheel-joint parameters, read from the env's model.
# (render_mode="human" doesn't open a window until you call env.render().)
lm   = env.left_motor_id
ljid = mujoco.mj_name2id(env.model, mujoco.mjtObj.mjOBJ_JOINT, "left_wheel_joint")
ldof = env.model.jnt_dofadr[ljid]

tau_stall    = float(env.model.actuator_gainprm[lm, 0])
b_m          = abs(float(env.model.actuator_biasprm[lm, 2]))
frictionloss = float(env.model.dof_frictionloss[ldof])
damping      = float(env.model.dof_damping[ldof])
armature     = float(env.model.dof_armature[ldof])

# Wheel radius from the cylinder collider
wid = mujoco.mj_name2id(env.model, mujoco.mjtObj.mjOBJ_GEOM, "wheel_left_col")
r_wheel = float(env.model.geom_size[wid, 0])

omega_nl_pred = (tau_stall - frictionloss) / b_m

print(f"tau_stall (gainprm)   = {tau_stall:.4f} N-m")
print(f"b_m       (-biasprm2) = {b_m:.5f} N-m/(rad/s)")
print(f"frictionloss          = {frictionloss:.4f} N-m")
print(f"joint damping         = {damping:.5f}   <-- must be ~0")
print(f"armature              = {armature:.2e} kg-m^2")
print(f"wheel radius          = {r_wheel:.5f} m")
print(f"\npredicted omega_nl    = {omega_nl_pred:.2f} rad/s "
      f"({omega_nl_pred*60/(2*math.pi):.0f} RPM)")
print(f"predicted top speed   = {omega_nl_pred*r_wheel:.3f} m/s")
print(f"predicted tau_m = J/b = {armature/b_m*1000:.0f} ms")

check("joint damping is ~0 (else double-counts biasprm[2])", damping < 1e-9,
      f"damping={damping}")
check("no-load constraint satisfied",
      abs(tau_stall - b_m*omega_nl_pred - frictionloss) < 1e-6)
check("motor can actually turn (stall torque > friction)", tau_stall > frictionloss)

In [ ]:
# Spin a free wheel up at full duty and check where it settles.
# Gravity is temporarily zeroed and the chassis lifted, then restored.
g_saved = env.model.opt.gravity.copy()
try:
    env.reset(seed=0)
    env.model.opt.gravity[:] = 0.0
    env.data.qpos[2] = 1.0          # lift clear of the floor
    env.data.qvel[:] = 0.0
    mujoco.mj_forward(env.model, env.data)

    vels, times = [], []
    for _ in range(4000):           # 20 s at 5 ms
        env.data.ctrl[env.left_motor_id] = 1.0
        mujoco.mj_step(env.model, env.data)
        vels.append(env.data.sensor(env.sensor_left_wheel_vel).data[0])
        times.append(env.data.time)
finally:
    env.model.opt.gravity[:] = g_saved
    env.reset(seed=0)

vels, times = np.array(vels), np.array(times)
omega_measured = vels[-500:].mean()

print(f"measured omega_nl = {omega_measured:.3f} rad/s   (predicted {omega_nl_pred:.3f})")
check("free-spin speed matches prediction",
      abs(omega_measured - omega_nl_pred) / omega_nl_pred < 0.05,
      f"{100*abs(omega_measured-omega_nl_pred)/omega_nl_pred:.1f}% error")

idx = np.argmax(vels > 0.632 * omega_measured)
print(f"measured tau_m    = {times[idx]*1000:.0f} ms   "
      f"(predicted J/b = {armature/b_m*1000:.0f} ms)")

In [ ]:
omegas = np.linspace(0, omega_nl_pred*1.1, 50)
fig, ax = plt.subplots(figsize=(6, 3.6))
for u in [0.25, 0.5, 0.75, 1.0]:
    ax.plot(omegas, tau_stall*u - b_m*omegas, label=f"duty={u}")
ax.axhline(0, color="k", lw=0.6)
ax.axhline(frictionloss, color="r", ls=":", lw=1, label="frictionloss")
ax.axvline(omega_nl_pred, color="g", ls="--", lw=1, label="omega_nl")
ax.set_xlabel("wheel speed [rad/s]"); ax.set_ylabel("torque [N-m]")
ax.set_title("FM90 torque-speed (model)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("The full-duty line should cross frictionloss exactly at omega_nl.")
print(f"Speed-limited, not torque-limited: wheels saturate at "
      f"{omega_nl_pred*r_wheel:.2f} m/s.")

## Pitch trim test

In [ ]:
mujoco.mj_resetData(env.model, env.data)

# set the KNOWN-stable equilibrium pose: chassis +11.4
th = math.radians(11.4)
env.data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
env.data.qvel[:] = 0
mujoco.mj_forward(env.model, env.data)

# let it settle a bit to confirm it's actually the stable point
for _ in range(50):
    env.data.ctrl[:] = 0
    mujoco.mj_step(env.model, env.data)
w,x,y,z = env.data.sensor("imu_orientation").data
pt = math.degrees(math.atan2(1-2*(x*x+y*y), -2*(y*z+w*x)))
print(f"env pitch_true at equilibrium: {pt:+.2f} deg")

In [ ]:
m, d = env.model, env.data
mujoco.mj_resetData(m, d)
th = math.radians(11.4)                      # the CONFIRMED stable chassis pose
d.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
d.qvel[:] = 0
mujoco.mj_forward(m, d)                       # read at the exact pose, no stepping
w,x,y,z = d.sensor("imu_orientation").data
pt = math.degrees(math.atan2(1-2*(x*x+y*y), -2*(y*z+w*x)))
print(f"pitch_true at the +11.4 chassis equilibrium: {pt:+.2f} deg")

In [ ]:
env = BalanceBotEnv(mjcf_path=MJCF_PATH, chassis_trim_deg=11.4)
# temporarily instrument: manually walk reset's tail
super(type(env), env).reset(seed=0)
mujoco.mj_resetData(env.model, env.data)
env.data.qpos[3:7] = [math.cos(env._eq_chassis_rad/2), 0, math.sin(env._eq_chassis_rad/2), 0]
env._pitch = env._pitch_trim_rad
env._vel_est = 0.0
mujoco.mj_forward(env.model, env.data)
print(f"before _get_obs: self._pitch = {math.degrees(env._pitch):+.2f}")
_, ay, az = env.data.sensor("imu_accel").data
print(f"  accel_y={ay:.3f} accel_z={az:.3f} accel_pitch={math.degrees(math.atan2(az,-ay)):+.2f}")
obs = env._get_obs()
print(f"after _get_obs:  self._pitch = {math.degrees(env._pitch):+.2f}, obs[0]={math.degrees(obs[0]):+.2f}")

## Other tests

In [ ]:
env = BalanceBotEnv(mjcf_path=MJCF_PATH, chassis_trim_deg=CHASSIS_PITCH_TRIM_DEG)
lengths = []
env.reset(seed=0); ep = 0
for _ in range(5000):
    _, _, term, trunc, _ = env.step(env.action_space.sample())
    ep += 1
    if term or trunc:
        lengths.append(ep); ep = 0; env.reset()
print(f"{len(lengths)} episodes, mean length {np.mean(lengths):.0f}, max {np.max(lengths)}")

In [ ]:
env = BalanceBotEnv(mjcf_path=MJCF_PATH, chassis_trim_deg=11.4)
mujoco.mj_resetData(env.model, env.data)
env.data.qvel[5] = 1.0          # +1 rad/s yaw (about world/chassis Z)
mujoco.mj_forward(env.model, env.data)
print("gyro at pure yaw:", np.round(env.data.sensor("imu_gyro").data, 3))

In [ ]:
env = BalanceBotEnv(mjcf_path=MJCF_PATH, chassis_trim_deg=11.4)
env.reset(seed=0)
for _ in range(200):
    env.step(env.action_space.sample())
print(f"pos_est={env._pos_est:.4f}  heading_est={env._heading_est:.4f}  vel_est={env._vel_est:.4f}")

In [ ]:
env = BalanceBotEnv(mjcf_path=MJCF_PATH, chassis_trim_deg=11.4)
obs, _ = env.reset(seed=0)
print("obs:", obs, "shape:", obs.shape)   # (6,), 5th element is cmd_pos (~0 at reset)
env.step(env.action_space.sample())
print("after step, cmd_pos:", env._cmd_pos)   # should be small nonzero

In [6]:
env = BalanceBotEnv(mjcf_path=MJCF_PATH, chassis_trim_deg=11.4,
                    domain_rand=DomainRandomConfig(init_pitch_range_deg=5.0, init_pitch_rate_range=0.8))
for seed in range(20):
    obs, _ = env.reset(seed=seed)
    print(f"seed {seed}: start pitch={math.degrees(obs[0]):+.1f} deg, pitch_rate={obs[1]:+.2f}")

eq=11.4 dr_range=5.0 start_chassis=12.8
seed 0: start pitch=-14.0 deg, pitch_rate=+0.83
eq=11.4 dr_range=5.0 start_chassis=11.5
seed 1: start pitch=-13.1 deg, pitch_rate=-0.36
eq=11.4 dr_range=5.0 start_chassis=9.0
seed 2: start pitch=-10.5 deg, pitch_rate=+0.01
eq=11.4 dr_range=5.0 start_chassis=7.3
seed 3: start pitch=-8.7 deg, pitch_rate=+0.12
eq=11.4 dr_range=5.0 start_chassis=15.8
seed 4: start pitch=-17.4 deg, pitch_rate=-0.49
eq=11.4 dr_range=5.0 start_chassis=14.5
seed 5: start pitch=-16.0 deg, pitch_rate=-0.51
eq=11.4 dr_range=5.0 start_chassis=11.8
seed 6: start pitch=-13.1 deg, pitch_rate=+0.38
eq=11.4 dr_range=5.0 start_chassis=12.7
seed 7: start pitch=-14.4 deg, pitch_rate=-0.91
eq=11.4 dr_range=5.0 start_chassis=9.7
seed 8: start pitch=-11.3 deg, pitch_rate=-0.60
eq=11.4 dr_range=5.0 start_chassis=15.1
seed 9: start pitch=-16.5 deg, pitch_rate=+0.24
eq=11.4 dr_range=5.0 start_chassis=16.0
seed 10: start pitch=-17.3 deg, pitch_rate=+0.14
eq=11.4 dr_range=5.0 start_chassis=

In [9]:
# Fall test
env = BalanceBotEnv(mjcf_path=MJCF_PATH, chassis_trim_deg=11.4)
obs, _ = env.reset(seed=0)

# Force the robot to a known start angle matching the hardware drop (e.g. near upright)
# Set chassis to ~0 deg (upright) to match a hardware release from upright:
start = math.radians(2.0)   # match whatever angle you released from on hardware
env.data.qpos[3:7] = [math.cos(start/2), 0, math.sin(start/2), 0]
env.data.qvel[:] = 0        # start at rest, like a clean release
env._pitch = -start
mujoco.mj_forward(env.model, env.data)

print("t,pitch_deg,pitch_rate")
for i in range(120):        # ~0.6s at 5ms
    obs, _, term, trunc, _ = env.step(np.array([0.0, 0.0]))   # ZERO action - passive fall
    t = i * 0.005
    print(f"{t:.3f},{math.degrees(obs[0]):.2f},{obs[1]:.3f}")
    if term or trunc: break

eq=11.4 dr_range=none start_chassis=11.4
t,pitch_deg,pitch_rate
0.000,-1.95,0.000
0.005,-1.99,0.000
0.010,-2.01,0.084
0.015,-2.01,0.151
0.020,-1.99,0.209
0.025,-1.95,0.264
0.030,-1.90,0.316
0.035,-1.84,0.368
0.040,-1.76,0.420
0.045,-1.66,0.472
0.050,-1.55,0.525
0.055,-1.43,0.578
0.060,-1.29,0.632
0.065,-1.14,0.688
0.070,-0.97,0.744
0.075,-0.78,0.801
0.080,-0.58,0.859
0.085,-0.36,0.919
0.090,-0.12,0.980
0.095,0.13,1.042
0.100,0.40,1.106
0.105,0.69,1.171
0.110,1.00,1.238
0.115,1.32,1.307
0.120,1.66,1.377
0.125,2.03,1.450
0.130,2.41,1.524
0.135,2.82,1.601
0.140,3.24,1.680
0.145,3.69,1.761
0.150,4.16,1.845
0.155,4.65,1.931
0.160,5.17,2.021
0.165,5.71,2.113
0.170,6.27,2.208
0.175,6.86,2.306
0.180,7.48,2.408
0.185,8.13,2.513
0.190,8.80,2.622
0.195,9.51,2.735
0.200,10.25,2.851
0.205,11.01,2.972
0.210,11.81,3.098
0.215,12.65,3.228
0.220,13.52,3.363
0.225,14.43,3.503
0.230,15.37,3.649
0.235,16.36,3.800
